# ChronoZ Benchmark Plotting — Selected Delta Models

This notebook visualizes each of the 36 selected series using the transformation and research-use model family recorded in `notebook/deltaAnalysis/CURRENT_MODEL_NOTES.md`. It does not alter cleaned data or claim production validation.

**Plot convention:** transform the series first, center the transformed observations around a reference mean of zero, then overlay the reference scale as horizontal ±1σ and ±2σ bands. For robust models, use median/MAD-based modified z-scores. The plotted zero is a neutral *statistical reference*, not necessarily a healthy/ideal economic state.

Native frequencies are preserved and changes across missing calendar periods are skipped. Regime/structural labels are exploratory proxies because validated regime boundaries are not defined in the selection notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('data/cleaned'), Path('../data/cleaned')]
cleaned_dir = next((p for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Run from repository root or notebook/.')

# Final selection from CURRENT_MODEL_NOTES.md (2026-09-20).
selection = {
 'HOUST':('log','GOOD'),'PERMIT':('log','GOOD'),'JTSJOL':('log','GOOD'),'OPHNFB':('log','GOOD'),'PPIFID':('log','GOOD'),
 'DRSDCILM':('level','GOOD'),'DRTSCILM':('level','GOOD'),'IMPGSC1':('log','GOOD'),'EXPGSC1':('log','GOOD'),
 'PCEPILFE':('log','GOOD'),'PCEPI':('log','GOOD'),'CPIAUCSL':('log','GOOD'),'CPILFESL':('log','GOOD'),
 'DRBLACBS':('level','GOOD'),'TCU':('level','GOOD'),'CIVPART':('level','GOOD'),'FEDFUNDS':('level','GOOD'),
 'PPIACO':('log','REGIME-SENSITIVE'),'PCEC96':('log','ROBUST-CALIBRATION'),'W875RX1':('log','GOOD'),
 'RRSFS':('log','GOOD'),'CES0500000003':('log','GOOD'),'INDPRO':('log','GOOD'),'IPMAN':('log','GOOD'),
 'TOTALSL':('log','GOOD'),'BUSLOANS':('log','GOOD'),'M2SL':('log','GOOD'),'AWHAETP':('level','DISCRETE'),
 'AWHMAN':('level','GOOD'),'RPI':('log','GOOD'),'PAYEMS':('log','GOOD'),'ICSA':('log','ROBUST'),
 'WALCL':('log','REGIME-ROBUST'),'WRESBAL':('log','REGIME-ROBUST'),'TOTRESNS':('log','STRUCTURAL-ROBUST'),
 'UNRATE':('level','DISCRETE-SHOCK')
}

def read_series(series_id):
    path = cleaned_dir / f'{series_id}.csv'
    if not path.exists(): return None
    df = pd.read_csv(path)
    if 'value' not in df: return None
    value = pd.to_numeric(df['value'], errors='coerce')
    if 'date' in df.columns:
        dates = pd.to_datetime(df['date'], errors='coerce')
    elif 'observation_date' in df.columns:
        dates = pd.to_datetime(df['observation_date'], errors='coerce')
    else:
        dates = pd.Series(pd.RangeIndex(len(df)))
    out = pd.DataFrame({'date':dates, 'value':value}).dropna().sort_values('date').drop_duplicates('date').set_index('date')
    return out['value']

def transform_change(series, transform):
    # Keep changes only when consecutive observations match native calendar steps.
    delta = series.diff()
    dates = series.index
    if isinstance(dates, pd.DatetimeIndex) and len(dates) > 1:
        diffs = pd.Series(dates[1:] - dates[:-1], index=dates[1:])
        month_step = (dates.to_series().diff().dt.days <= 45) & (dates.to_series().diff().dt.days >= 20)
        quarter_step = (dates.to_series().diff().dt.days <= 120) & (dates.to_series().diff().dt.days >= 70)
        weekly_step = (dates.to_series().diff().dt.days <= 14) & (dates.to_series().diff().dt.days >= 4)
        valid_step = month_step | quarter_step | weekly_step
        delta = delta.where(valid_step)
    if transform == 'log':
        if (series.dropna() <= 0).any():
            raise ValueError('Log change requested for non-positive series')
        change = 100 * (np.log(series) - np.log(series.shift(1)))
        change = change.where(delta.notna())
    else:
        change = delta
    return change.replace([np.inf, -np.inf], np.nan).dropna()

summary_rows = []
plot_data = {}
for sid, (transform, family) in selection.items():
    levels = read_series(sid)
    if levels is None or levels.empty: continue
    change = transform_change(levels, transform)
    if change.empty: continue
    # PCEC96 calibration: modified-z rule using median/MAD, keep all points but calibrate on unflagged history.
    center, scale = change.mean(), change.std(ddof=1)
    flagged = pd.Series(False, index=change.index)
    if family == 'ROBUST-CALIBRATION':
        med, mad = change.median(), (change - change.median()).abs().median()
        if mad > 0: flagged = (0.6745 * (change - med) / mad).abs() > 3.5
        eligible = change[~flagged]
        if len(eligible) >= 2: center, scale = eligible.mean(), eligible.std(ddof=1)
    # Alternative robust families: robust center/scale for visualization; no observations removed.
    if family in ('ROBUST','REGIME-ROBUST','STRUCTURAL-ROBUST','DISCRETE-SHOCK'):
        center = change.median()
        mad = (change - center).abs().median()
        scale = mad / 0.6745 if mad > 0 else change.std(ddof=1)
    # Exploratory rolling regime proxy for regime-based series (not an approved regime classification).
    score = (change - center) / scale if pd.notna(scale) and scale > 0 else pd.Series(np.nan, index=change.index)
    plot_data[sid] = (change, score, family, flagged)
    summary_rows.append({'series_id':sid,'transformation':'100*Δln(x)' if transform=='log' else 'Δ level','model_family':family,'n_changes':len(change),'reference_center':center,'reference_scale':scale,'flagged_n':int(flagged.sum())})

model_summary = pd.DataFrame(summary_rows).sort_values('series_id').reset_index(drop=True)
display(model_summary.round(5))
print(f'Series plotted: {len(plot_data)} / {len(selection)} selected')

## Per-series benchmark plots

Each figure has two panels: (1) transformed changes centered on the model reference with ±1 and ±2 scale bands; (2) standardized/robust scores through time with a zero line and ±1/±2 reference bands. Discrete-model series also show their empirical change frequencies. Tail observations are retained. Regime/structural models use a robust global proxy in this plotting notebook until explicit, validated regime/epoch boundaries are specified.

In [ ]:
plot_dir = Path('outputs/benchmark/plots') if Path('outputs').exists() else Path('../outputs/benchmark/plots')
plot_dir.mkdir(parents=True, exist_ok=True)
for sid, (change, score, family, flagged) in plot_data.items():
    center = change.median() if family in ('ROBUST','REGIME-ROBUST','STRUCTURAL-ROBUST','DISCRETE-SHOCK') else (change[~flagged].mean() if family=='ROBUST-CALIBRATION' and (~flagged).sum() else change.mean())
    scale = (change-center).abs().median()/0.6745 if family in ('ROBUST','REGIME-ROBUST','STRUCTURAL-ROBUST','DISCRETE-SHOCK') and (change-center).abs().median()>0 else change.std(ddof=1)
    if family=='ROBUST-CALIBRATION':
        cal=change[~flagged]; center=cal.mean() if len(cal) else change.mean(); scale=cal.std(ddof=1) if len(cal)>1 else change.std(ddof=1)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
    axes[0].plot(change.index, change-center, linewidth=1, label='Centered change')
    for k, alpha in [(1,0.65),(2,0.35)]:
        axes[0].axhline(k*scale, linestyle='--', linewidth=1, alpha=alpha, label=f'+{k} scale')
        axes[0].axhline(-k*scale, linestyle='--', linewidth=1, alpha=alpha, label=f'-{k} scale')
    axes[0].axhline(0, color='black', linewidth=1); axes[0].set_title(f'{sid} — transformed change | {family}'); axes[0].set_ylabel('Change − reference center'); axes[0].legend(ncol=3, fontsize=8)
    axes[1].plot(score.index, score, linewidth=1, label='Reference score')
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].axhline(1, linestyle='--', alpha=.6); axes[1].axhline(-1, linestyle='--', alpha=.6)
    axes[1].axhline(2, linestyle=':', alpha=.6); axes[1].axhline(-2, linestyle=':', alpha=.6)
    if flagged.any(): axes[1].scatter(change.index[flagged], score.loc[flagged], marker='x', label='Excluded from calibration, still scored')
    axes[1].set_title('Standardized movement relative to selected reference'); axes[1].set_ylabel('Score'); axes[1].legend(fontsize=8)
    axes[1].set_xlabel('Observation date')
    fig.savefig(plot_dir / f'{sid}_benchmark.png', dpi=130); plt.show(); plt.close(fig)

# Empirical change-frequency plots for discrete-change selections.
for sid in ['AWHAETP','UNRATE']:
    if sid not in plot_data: continue
    change = plot_data[sid][0]
    freq = change.value_counts(normalize=True).sort_index() * 100
    ax = freq.plot(kind='bar', figsize=(10,4), title=f'{sid} — empirical discrete change frequencies')
    ax.set_xlabel('Observed change'); ax.set_ylabel('Frequency (%)'); plt.tight_layout()
    plt.savefig(plot_dir / f'{sid}_discrete_frequency.png', dpi=130); plt.show(); plt.close()

model_summary.to_csv(plot_dir.parent / 'selected_delta_model_statistics.csv', index=False)
print(f'Plots and model statistics saved to {plot_dir.resolve()}')